# Long-term climate wildfire risk projection

This notebook calculates a long-term wildfire risk projection for Deliblato Sands using:

- static fire susceptibility raster
- NASA GDDP-CMIP6 climate projections
- historical baseline climate
- future climate scenario
- protected-area averaged climate-change signal
- climate modifier applied to the static fire-risk layer

The result is a climate tendency layer, not a day-specific forecast.

## 1. Install packages if needed

Run this in Anaconda Prompt if the packages are missing:

```bash
conda install -c conda-forge earthengine-api geemap
```

In [1]:
import ee
import geemap

## 2. Authenticate and initialize Earth Engine

In [2]:
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    # CHANGE PROJECT NAME
    ee.Initialize(project='ee-minucher')

## 3. Load assets

In [3]:
roi = ee.FeatureCollection("projects/ee-minucher/assets/DeliblatoROI_SQFIN")
roisrp = ee.FeatureCollection("projects/ee-minucher/assets/SRPDeliblatoWGS84")
stat4 = ee.Image("projects/ee-minucher/assets/StaticV4")

roi_geom = roi.geometry()
roisrp_geom = roisrp.geometry()

## 4. Load NASA GDDP-CMIP6

In [4]:
cmip6 = ee.ImageCollection("NASA/GDDP-CMIP6")

print("CMIP6 image count:", cmip6.size().getInfo())
print("Example image:")
print(cmip6.first().getInfo()["properties"])
print("Example bands:")
cmip6.first().bandNames().getInfo()

CMIP6 image count: 2997564
Example image:
{'year': 1950, 'system:time_end': -631065600000, 'version': 1.1, 'system:time_start': -631152000000, 'license': 'CC-BY-4.0', 'month': 1, 'scenario': 'historical', 'system:footprint': {'type': 'LinearRing', 'coordinates': [[-180, -90], [180, -90], [180, 90], [-180, 90], [-180, -90]]}, 'model': 'ACCESS-CM2', 'system:asset_size': 10774567, 'day': 1, 'system:index': 'historical_ACCESS-CM2_19500101'}
Example bands:


['hurs', 'huss', 'pr', 'rlds', 'rsds', 'sfcWind', 'tas', 'tasmax', 'tasmin']

## 5. User settings

In [5]:
# Climate settings
scenario = "ssp245"     # options commonly used here: "ssp245" or "ssp585"
month = 8               # 1-12

# Historical baseline and future projection period
hist_start = "1995-01-01"
hist_end   = "2014-12-31"

future_start = "2031-01-01"
future_end   = "2041-12-31"

# Analysis scale for NASA GDDP-CMIP6
scale = 27830

print("Scenario:", scenario)
print("Month:", month)
print("Historical period:", hist_start, "to", hist_end)
print("Future period:", future_start, "to", future_end)

Scenario: ssp245
Month: 8
Historical period: 1995-01-01 to 2014-12-31
Future period: 2031-01-01 to 2041-12-31


## 6. Helper functions

In [6]:
def mean_over_area(img, geom, scale=27830):
    result = img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=geom,
        scale=scale,
        bestEffort=True,
        maxPixels=1e9,
    )
    return ee.Number(result.values().get(0))


def band_monthly_ensemble_median(scenario_name, start_date, end_date, band_name, month_num):
    '''
    For one CMIP6 variable:
    1. filter by scenario, period and month
    2. calculate mean image for each model
    3. calculate ensemble median across models
    '''
    base = (
        cmip6
        .filterDate(start_date, end_date)
        .filter(ee.Filter.eq("scenario", scenario_name))
        .filter(ee.Filter.calendarRange(month_num, month_num, "month"))
        .filter(ee.Filter.listContains("system:band_names", band_name))
        .select(band_name)
    )

    models = ee.List(base.aggregate_array("model")).distinct()

    per_model_means = ee.ImageCollection.fromImages(
        models.map(
            lambda model: (
                base
                .filter(ee.Filter.eq("model", ee.String(model)))
                .mean()
                .rename(band_name)
                .set("model", model)
            )
        )
    )

    return per_model_means.median().rename(band_name)

## 7. Build historical monthly ensemble climate

In [7]:
hist_tas = (
    band_monthly_ensemble_median("historical", hist_start, hist_end, "tas", month)
    .subtract(273.15)
    .rename("hist_air_temperature_C")
)

hist_pr = (
    band_monthly_ensemble_median("historical", hist_start, hist_end, "pr", month)
    .multiply(86400)
    .rename("hist_precipitation_mm_day")
)

hist_hurs = (
    band_monthly_ensemble_median("historical", hist_start, hist_end, "hurs", month)
    .rename("hist_relative_humidity_percent")
)

hist_rsds = (
    band_monthly_ensemble_median("historical", hist_start, hist_end, "rsds", month)
    .rename("hist_solar_radiation_Wm2")
)

hist_wind = (
    band_monthly_ensemble_median("historical", hist_start, hist_end, "sfcWind", month)
    .rename("hist_wind_speed_ms")
)

## 8. Build future monthly ensemble climate

In [8]:
fut_tas = (
    band_monthly_ensemble_median(scenario, future_start, future_end, "tas", month)
    .subtract(273.15)
    .rename("future_air_temperature_C")
)

fut_pr = (
    band_monthly_ensemble_median(scenario, future_start, future_end, "pr", month)
    .multiply(86400)
    .rename("future_precipitation_mm_day")
)

fut_hurs = (
    band_monthly_ensemble_median(scenario, future_start, future_end, "hurs", month)
    .rename("future_relative_humidity_percent")
)

fut_rsds = (
    band_monthly_ensemble_median(scenario, future_start, future_end, "rsds", month)
    .rename("future_solar_radiation_Wm2")
)

fut_wind = (
    band_monthly_ensemble_median(scenario, future_start, future_end, "sfcWind", month)
    .rename("future_wind_speed_ms")
)

## 9. Area-averaged historical and future values

In [10]:
# Calculation (takes longer) 

hist_tas_mean  = mean_over_area(hist_tas, roisrp_geom, scale)
hist_pr_mean   = mean_over_area(hist_pr, roisrp_geom, scale)
hist_hurs_mean = mean_over_area(hist_hurs, roisrp_geom, scale)
hist_rsds_mean = mean_over_area(hist_rsds, roisrp_geom, scale)
hist_wind_mean = mean_over_area(hist_wind, roisrp_geom, scale)

fut_tas_mean  = mean_over_area(fut_tas, roisrp_geom, scale)
fut_pr_mean   = mean_over_area(fut_pr, roisrp_geom, scale)
fut_hurs_mean = mean_over_area(fut_hurs, roisrp_geom, scale)
fut_rsds_mean = mean_over_area(fut_rsds, roisrp_geom, scale)
fut_wind_mean = mean_over_area(fut_wind, roisrp_geom, scale)

values = {
    "Historical air temperature (C)": hist_tas_mean.getInfo(),
    "Future air temperature (C)": fut_tas_mean.getInfo(),
    "Historical precipitation (mm/day)": hist_pr_mean.getInfo(),
    "Future precipitation (mm/day)": fut_pr_mean.getInfo(),
    "Historical relative humidity (%)": hist_hurs_mean.getInfo(),
    "Future relative humidity (%)": fut_hurs_mean.getInfo(),
    "Historical solar radiation (W/m2)": hist_rsds_mean.getInfo(),
    "Future solar radiation (W/m2)": fut_rsds_mean.getInfo(),
    "Historical wind speed (m/s)": hist_wind_mean.getInfo(),
    "Future wind speed (m/s)": fut_wind_mean.getInfo(),
}

for k, v in values.items():
    print(f"{k}: {v:.2f}")

Historical air temperature (C): 23.66
Future air temperature (C): 25.47
Historical precipitation (mm/day): 1.75
Future precipitation (mm/day): 1.68
Historical relative humidity (%): 69.73
Future relative humidity (%): 67.06
Historical solar radiation (W/m2): 248.65
Future solar radiation (W/m2): 253.12
Historical wind speed (m/s): 1.95
Future wind speed (m/s): 1.96


## 10. Calculate climate-change signals

In [ ]:
d_tas = fut_tas_mean.subtract(hist_tas_mean)        
d_hurs = hist_hurs_mean.subtract(fut_hurs_mean)    
d_pr = hist_pr_mean.subtract(fut_pr_mean)          
d_rsds = fut_rsds_mean.subtract(hist_rsds_mean)
d_wind = fut_wind_mean.subtract(hist_wind_mean)    

deltas = {
    "Temperature change (C)": d_tas.getInfo(),
    "Humidity drying signal (%)": d_hurs.getInfo(),
    "Precipitation drying signal (mm/day)": d_pr.getInfo(),
    "Solar radiation change (W/m2)": d_rsds.getInfo(),
    "Wind speed change (m/s)": d_wind.getInfo(),
}

for k, v in deltas.items():
    print(f"{k}: {v:.2f}")

## 11. Convert climate signal to modifier

In [ ]:
# Normalize each climate-change signal to approximately -1 to +1.
# Divisors are expert scaling parameters and can be tuned later.

t_term = ee.Image.constant(d_tas.divide(4)).clamp(-1, 1)
hurs_term = ee.Image.constant(d_hurs.divide(20)).clamp(-1, 1)
pr_term = ee.Image.constant(d_pr.divide(3)).clamp(-1, 1)
rsds_term = ee.Image.constant(d_rsds.divide(40)).clamp(-1, 1)
wind_term = ee.Image.constant(d_wind.divide(2)).clamp(-1, 1)

climate_signal = (
    t_term.multiply(0.35)
    .add(hurs_term.multiply(0.20))
    .add(pr_term.multiply(0.20))
    .add(wind_term.multiply(0.15))
    .add(rsds_term.multiply(0.10))
    .rename("climate_signal")
)

# Modifier centered on 1.0.
# 1.0 = no change relative to historical climate.
# >1.0 = increased fire-prone climate tendency.
# <1.0 = reduced fire-prone climate tendency.

climate_modifier = (
    ee.Image(1)
    .add(climate_signal.multiply(0.5))
    .clamp(0.7, 1.5)
    .rename("climate_modifier")
)

## 12. Calculate long-term climate fire risk

In [ ]:
fire_risk_climate = (
    stat4
    .multiply(climate_modifier)
    .clamp(0, 100)
    .rename("long_term_climate_fire_risk")
    .updateMask(stat4.gt(0))
)

## 13. Diagnostics

In [ ]:
def area_mean(img, name):
    value = mean_over_area(img.rename("x"), roisrp_geom, scale).getInfo()
    print(f"{name}: {value:.2f}")

area_mean(climate_signal, "Climate signal")
area_mean(climate_modifier, "Climate modifier")
area_mean(fire_risk_climate, "Long-term climate fire risk")

## 14. Interactive map

In [ ]:
risk_palette = [
    "#006400", "#228B22", "#7FBF3F",
    "#ADFF2F", "#FFFF00", "#FFD700",
    "#FFA500", "#FF7F00", "#FF4500",
    "#8B0000",
]

m = geemap.Map()
m.centerObject(roisrp, 10)
m.add_basemap("HYBRID")

m.addLayer(
    fire_risk_climate.clip(roi_geom),
    {"min": 0, "max": 100, "palette": risk_palette},
    f"Long-term climate fire risk {scenario}, month {month}, 2031-2041",
)

m.addLayer(
    climate_modifier.clip(roi_geom),
    {"min": 0.7, "max": 1.5, "palette": ["blue", "white", "red"]},
    "Climate modifier",
    shown=False,
)

m.addLayer(
    ee.Image().paint(roisrp, 1, 2),
    {"palette": ["00FFFF"]},
    "Protected area boundary",
)

m.addLayerControl()
m

## 15. Optional export

In [ ]:
# Uncomment to export the result as GeoTIFF.
# This may take some time depending on the region and scale.

# geemap.ee_export_image(
#     fire_risk_climate.clip(roi_geom),
#     filename=f"outputs/long_term_climate_fire_risk_{scenario}_month_{month}_2031_2041.tif",
#     scale=100,
#     region=roi_geom,
#     file_per_band=False,
# )